# LoRA MLP + Attention (Low Rank)

This notebook mirrors the full-run LoRA pipeline and focuses on low-rank target-module ablations for attention-only vs attention+MLP adapters.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import torch
from IPython.display import display

## Colab setup and project root

In [2]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    CANDIDATE_ROOT = Path('/content/drive/MyDrive/DL_Final_Project')
    PROJECT_ROOT = CANDIDATE_ROOT if CANDIDATE_ROOT.exists() else Path.cwd()
else:
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks':
        PROJECT_ROOT = cwd.parent
    elif cwd.name == 'LoRA_study' and cwd.parent.name == 'notebooks':
        PROJECT_ROOT = cwd.parent.parent
    else:
        PROJECT_ROOT = cwd

print('IN_COLAB:', IN_COLAB)
print('PROJECT_ROOT:', PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB: True
PROJECT_ROOT: /content/drive/MyDrive/DL_Final_Project


In [3]:
import importlib.metadata as md
import importlib.util
import subprocess


def _parse_version_tuple(version_str: str) -> tuple[int, ...]:
    parts = []
    for token in version_str.split('.'):
        digits = ''.join(ch for ch in token if ch.isdigit())
        parts.append(int(digits) if digits else 0)
    return tuple(parts)


required_torchao = (0, 16, 0)
if importlib.util.find_spec('torchao') is not None:
    v_str = md.version('torchao')
    print(f'Found torchao=={v_str}')
    if _parse_version_tuple(v_str) < required_torchao:
        print('Uninstalling incompatible torchao...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
        raise SystemExit('Removed incompatible torchao. Restart runtime and re-run from the top.')
    print('torchao is compatible.')
else:
    print('torchao is not installed (this is fine).')

torchao is not installed (this is fine).


In [4]:
from src.gen_lora_study_utils import (
    set_seed,
    load_split,
    apply_sanity_subset,
    cap_validation_rows,
)
from src.lora_best_candidates_full_run_utils import build_configs_from_named_space, run_lora_candidate_on_val
from src.lora_tuning_block_utils import count_trainable_params_for_config, run_test_submission_from_saved_adapter

## Runtime controls (same style as full-run notebook)

In [5]:
save = True

# For the sweep, keep this False so you do not run test inference for every config.
# After choosing the best validation config, rerun only that config with generate_submission=True.
generate_submission = True

submission_ablation_ids = ['score_dora_attn_mlp_r4_score_512_metadata_lightaug']

sanity_check = True
n = 2250  # ignored when sanity_check=False

seed = 42
max_val_examples = 256
top_k = 5
batch_size = 4
clear_cuda_between_runs = True

# For option_scoring-only runs, loss is a smoother early-stopping signal.
# For mixed option_scoring + SFT runs, use 'accuracy' instead; see note below.
early_stopping = None
epochs_per_val_check = None
early_stopping_metric = 'loss'
early_stopping_val_examples = 128 #None

DATA_DIR = PROJECT_ROOT / 'data'
IMAGE_DIR = DATA_DIR / 'images' / 'images'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'lora_mlp_attention_option_scoring'
ADAPTER_DIR = OUT_DIR / 'adapters'
MODEL_ID = 'HuggingFaceTB/SmolVLM-500M-Instruct'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

torch_dtype = (
    'bfloat16'
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else ('float16' if torch.cuda.is_available() else 'auto')
)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print({'save': save, 'generate_submission': generate_submission, 'sanity_check': sanity_check, 'n': n, 'device': DEVICE})
set_seed(seed)

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
{'save': True, 'generate_submission': True, 'sanity_check': True, 'n': 2250, 'device': 'cuda'}


In [6]:
train_df = load_split(DATA_DIR, 'train')
val_df = load_split(DATA_DIR, 'val')
test_df = load_split(DATA_DIR, 'test')

train_df = apply_sanity_subset(train_df, sanity_check, n, seed)
val_df = apply_sanity_subset(val_df, sanity_check, n, seed)
test_df = apply_sanity_subset(test_df, sanity_check, 1008, seed)
val_df = cap_validation_rows(val_df, max_val_examples)

print('train rows used:', len(train_df))
print('validation rows used:', len(val_df))
print('test rows used:', len(test_df))
print('IMAGE_DIR exists:', IMAGE_DIR.exists())

train rows used: 2250
validation rows used: 256
test rows used: 1008
IMAGE_DIR exists: True


## Candidate ablation spaces

The key differences are low-rank settings and target modules. Input/data ablations are toggled directly in each candidate config.

In [7]:
# Single best-attempt config after the SFT masking / generation / option-scoring fixes.

input_ablation = {
    # The strategy note says the original images are small and recommends trying bigger image size.
    # Use 512 for this run. If you later find many images are non-square, switch this to None
    # and control processor resolution instead.
    'image_size': 280,

    # Keep augmentation light. These images likely contain diagrams/text, so aggressive rotation/crop can hurt.
    'enable_image_augmentation': True,
    'brightness_jitter': 0.03,
    'slight_rotation_deg': 0.5,

    # Use metadata because subject/grade/topic may help route the reasoning.
    'include_metadata_in_prompt': True,
    'metadata_fields': ['subject', 'grade', 'topic'],
    'metadata_header': 'Metadata',

    # With output_format='letter_only', this is mostly harmless/unused,
    # but keep it simple in case other code references it.
    'answer_prefix_text': 'Answer:',
    'question_label': 'Question:',
    'choices_label': 'Choices:',
    'instruction_prefix': '',
}

prompt_structure = 'question_first'
context_mode = 'hint_lecture'
choice_format = 'letter_dot'

# Important: use letter-only with option scoring.
# This makes each candidate just "A", "B", "C", etc.
output_format = 'letter_only'

# Option scoring does not rely on free generation, but keep these sane for compatibility.
max_new_tokens = 4
decoding_strategy = 'greedy'
num_beams = 1
parse_rule = 'strict_first_letter'

# Option scoring is slower than SFT because it scores every candidate.
# Start with 12 rather than 25; early stopping can still stop sooner.
epochs = 4

single_lora_ablation_space = {
    'dora_attn_mlp_r4_score_512_metadata_lightaug': {
        # Main change: use multiple-choice candidate scoring rather than generative SFT.
        'training_objective': 'option_scoring',

        'prompt_structure': prompt_structure,
        'context_mode': context_mode,
        'choice_format': choice_format,
        'output_format': output_format,
        'max_new_tokens': max_new_tokens,
        'decoding_strategy': decoding_strategy,
        'num_beams': num_beams,
        'parse_rule': parse_rule,

        # DoRA + attention/MLP targets.
        # r=4 should stay safely below your 5M trainable-parameter cap while adapting both
        # attention behavior and internal MLP representations.
        'lora_r':8,
        'lora_alpha': 16,
        'lora_dropout': 0.03,
        'target_modules': [
            'q_proj', 'k_proj', 'v_proj', 'o_proj',
            'up_proj', 'down_proj', 'gate_proj',
        ],
        'lora_bias': 'none',
        'lora_task_type': 'CAUSAL_LM',
        'use_dora': False,
        'use_rslora': True,

        'epochs': epochs,

        # Your current option-scoring trainer is effectively row-by-row, so grad accumulation
        # may not be used unless you patched it in. Keep this at 1.
        'gradient_accumulation_steps': 1,

        # DoRA usually benefits from a slightly lower LR than vanilla LoRA.
        'lr': 5e-5,
        'weight_decay': 0.01,
        'max_grad_norm': 1.0,

        # These only matter if you added scheduler support.
        # If not, they will remain harmless config fields.
        'warmup_ratio': 0.05,
        'scheduler_type': 'cosine',

        **input_ablation,
    },
}

all_sft_configs = build_configs_from_named_space(single_lora_ablation_space, 'score')
display(pd.DataFrame(all_sft_configs))

,training_objective,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,num_beams,parse_rule,lora_r,...,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,metadata_header,answer_prefix_text,question_label,choices_label,instruction_prefix,ablation_name,ablation_id
0,option_scoring,question_first,hint_lecture,letter_dot,letter_only,4,greedy,1,strict_first_letter,8,...,0.5,True,"[subject, grade, topic]",Metadata,Answer:,Question:,Choices:,,dora_attn_mlp_r4_score_512_metadata_lightaug,score_dora_attn_mlp_r4_score_512_metadata_ligh...


In [8]:
fixed_context = {
    'MODEL_ID': MODEL_ID,
    'DEVICE': DEVICE,
    'torch_dtype': torch_dtype,
    'IMAGE_DIR': IMAGE_DIR,
    'train_df': train_df,
    'val_df': val_df,
    'test_df': test_df,
    'batch_size': batch_size,
    'clear_cuda_between_runs': clear_cuda_between_runs,
    'early_stopping': early_stopping,
    'epochs_per_val_check': epochs_per_val_check,
    'early_stopping_metric': early_stopping_metric,
    'early_stopping_val_examples': early_stopping_val_examples,
    # Save adapters once during val training so test submissions can reuse them.
    'save_trained_adapters': bool(generate_submission),
    'ADAPTER_DIR': ADAPTER_DIR,
}

_param_rows = []
for cfg in all_sft_configs:
    counts = count_trainable_params_for_config(cfg, {'MODEL_ID': MODEL_ID, 'DEVICE': DEVICE, 'torch_dtype': torch_dtype})
    _param_rows.append({
        'ablation_id': cfg['ablation_id'],
        'trainable_params': counts['trainable'],
        'total_params': counts['total'],
        'trainable_pct': round(float(counts['trainable_pct']), 4),
    })
display(pd.DataFrame(_param_rows).sort_values('ablation_id').reset_index(drop=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

,ablation_id,trainable_params,total_params,trainable_pct
0,score_dora_attn_mlp_r4_score_512_metadata_ligh...,4784128,512266432,0.9339


In [9]:
results = []
val_preds = []
histories = []
submission_frames = []

config_by_id = {cfg['ablation_id']: cfg for cfg in all_sft_configs}
requested_ids = [str(x) for x in (submission_ablation_ids or []) if str(x).strip()]
selected_ids = set(requested_ids) if requested_ids else set(config_by_id.keys())
missing_ids = sorted(set(requested_ids) - set(config_by_id.keys()))
if missing_ids:
    print('Skipping unknown ablation ids:', missing_ids)

for cfg in all_sft_configs:
    ablation_id = cfg['ablation_id']
    summary, val_pred_df, history_df = run_lora_candidate_on_val(config=cfg, fixed_context=fixed_context)
    results.append(summary)
    val_preds.append(val_pred_df)
    if len(history_df):
        histories.append(history_df)

    # Generate submission immediately after training for selected ablations (no retraining).
    if generate_submission and (ablation_id in selected_ids):
        adapter_path = ADAPTER_DIR / ablation_id
        test_pred_df, submission_df = run_test_submission_from_saved_adapter(
            config=cfg,
            fixed_context=fixed_context,
            adapter_dir=adapter_path,
            save=save,
            out_dir=OUT_DIR,
            filename_prefix=f'mlp_attn_{ablation_id}_',
        )
        _sub = submission_df.copy()
        _sub['ablation_id'] = ablation_id
        submission_frames.append(_sub)
        print(f'Generated submission without retraining for: {ablation_id}')
        display(test_pred_df.head())
        display(submission_df.head())

    if clear_cuda_between_runs and torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results).sort_values('accuracy', ascending=False).reset_index(drop=True)
val_predictions_df = pd.concat(val_preds, ignore_index=True) if val_preds else pd.DataFrame()
history_df = pd.concat(histories, ignore_index=True) if histories else pd.DataFrame()
display(results_df)

if submission_frames:
    combined_submission_preview = pd.concat(submission_frames, ignore_index=True)
    display(combined_submission_preview.head())

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Score epoch 1/4:   0%|          | 0/2250 [00:00<?, ?it/s]

Score epoch 2/4:   0%|          | 0/2250 [00:00<?, ?it/s]

Score epoch 3/4:   0%|          | 0/2250 [00:00<?, ?it/s]

Score epoch 4/4:   0%|          | 0/2250 [00:00<?, ?it/s]

...

Generated submission without retraining for: score_dora_attn_mlp_r4_score_512_metadata_lightaug


,id,pred_answer,num_choices
0,test_01918,3,4
1,test_02484,0,4
2,test_03843,2,4
3,test_02160,0,3
4,test_01805,0,2


,id,answer
0,test_01918,3
1,test_02484,0
2,test_03843,2
3,test_02160,0
4,test_01805,0


,training_objective,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,num_beams,parse_rule,lora_r,...,ablation_id,n_examples,accuracy,percent_correct,eval_loss,trainable_params,total_params,trainable_pct,train_seconds,adapter_dir
0,option_scoring,question_first,hint_lecture,letter_dot,letter_only,4,greedy,1,strict_first_letter,8,...,score_dora_attn_mlp_r4_score_512_metadata_ligh...,256,0.742188,74.21875,1.653327,4784128,512266432,0.933914,24254.309778,/content/drive/MyDrive/DL_Final_Project/output...


,id,answer,ablation_id
0,test_01918,3,score_dora_attn_mlp_r4_score_512_metadata_ligh...
1,test_02484,0,score_dora_attn_mlp_r4_score_512_metadata_ligh...
2,test_03843,2,score_dora_attn_mlp_r4_score_512_metadata_ligh...
3,test_02160,0,score_dora_attn_mlp_r4_score_512_metadata_ligh...
4,test_01805,0,score_dora_attn_mlp_r4_score_512_metadata_ligh...
